# 02 · Modern local options: embedding-based extraction + a small open LLM

Companion notebook to the MLforSEO tutorial *Automatically Generate Meta Descriptions Using Python: from BERT to LLMs*.

Two upgrades on the 2021 BERT summariser, both running **on your own machine or in Colab, no API key, no per-call cost**:

1. **Embedding-based extraction** with [`sentence-transformers`](https://www.sbert.net/). Same idea as the BERT summariser (pick the most representative sentence),
   but transparent and steerable. You decide what "representative" means: close to the whole page, close to the page title, and close to the query the page actually ranks for.
2. **Abstractive generation with a small open instruction-tuned LLM** ([Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) by default) that writes a new description from the page text.
   This reads better, but it *can* invent things. That's why every output goes through the same checks.

**Runtime:** the embedding model is small (about 90 MB) and fast on CPU. The 1.7B LLM is about 3.4 GB. Use a Colab T4 GPU or an Apple-silicon Mac (around 4 to 8 seconds per page in our tests); on a plain CPU it's slow.

In [1]:
# Install anything that's missing (works in Google Colab and locally)
import importlib.util, subprocess, sys

PACKAGES = {'sentence_transformers': 'sentence-transformers', 'transformers': 'transformers', 'accelerate': 'accelerate', 'torch': 'torch', 'nltk': 'nltk', 'trafilatura': 'trafilatura', 'pandas': 'pandas'}  # import name -> pip name
missing = [pip for mod, pip in PACKAGES.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Ready. Installed now:", missing or "nothing, all present")

Ready. Installed now: nothing, all present


## 1. Load your pages
Same input as notebook 01. If you have a **Top query** column (for example from Search Console via the *Search Analytics for Sheets* add-on), the extraction step uses it.

In [2]:
import pandas as pd

# OPTION A (the 2021 way): a CSV you exported from Screaming Frog, Sitebulb, etc.
#   Column A = URL, Column B = the page's main content. Title and Top query are optional extras.
INPUT_FILE = "inputdata.csv"

# OPTION B: no crawler export? Put one URL per line in urls.txt and set this to True.
#   trafilatura downloads each page and pulls out the main text and the title for you.
SCRAPE_FROM_URL_LIST = False
URL_LIST_FILE = "urls.txt"

# In Colab, upload your file first (folder icon on the left), or uncomment:
# from google.colab import files; files.upload()

if SCRAPE_FROM_URL_LIST:
    import trafilatura
    rows = []
    for url in [u.strip() for u in open(URL_LIST_FILE) if u.strip()]:
        html = trafilatura.fetch_url(url)
        text = trafilatura.extract(html) if html else None
        meta = trafilatura.extract_metadata(html) if html else None
        rows.append({"URL": url, "Content": text or "", "Title": getattr(meta, "title", "") or ""})
    df = pd.DataFrame(rows)
else:
    # error_bad_lines=False (2021) was removed in pandas 2.0; on_bad_lines="skip" replaces it
    df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig", on_bad_lines="skip")
    df = df.rename(columns={df.columns[0]: "URL", df.columns[1]: "Content"})

for col in ["Title", "Top query"]:
    if col not in df.columns:
        df[col] = ""
df = df.fillna("")
df["Words"] = df["Content"].str.split().str.len()
df[["URL", "Title", "Top query", "Words"]]

,URL,Title,Top query,Words
0,https://www.mlforseo.com/machine-learning-tuto...,"What Fuzzy Matching Is, and Where It Fits in SEO",what is fuzzy matching,2349
1,https://www.mlforseo.com/machine-learning-tuto...,Topic Modelling for SEO with BERTopic,bertopic seo,1571
2,https://www.mlforseo.com/machine-learning-impl...,How to do keyword clustering with KeyBERT,keybert keyword clustering,2212
3,https://www.mlforseo.com/machine-learning-impl...,How to Automatically Optimize your SEO Metadat...,automate meta descriptions,1938
4,https://www.mlforseo.com/glossary-term/bert-bi...,BERT (Bidirectional Encoder Representations fr...,what is bert,233


## 2. Embedding-based extraction

How it works:
1. Split the page into sentences and keep only **candidates** that could work as a description on their own: 70 to 155 characters, start with a capital letter,
   end with a full stop, and don't open with a word that needs context ("And", "This", "It"…).
2. Turn every candidate into an **embedding**, a list of numbers that captures meaning.
3. Score each candidate on three things: how close it is to the **page as a whole** (the average of all sentence embeddings), how close it is to the
   **focus** (the top query if you have one, otherwise the title), and a small bonus for appearing **early** on the page.
4. Keep the best one.

Change the weights to suit your site. `all-MiniLM-L6-v2` is English-only and fast. For other languages, try `paraphrase-multilingual-MiniLM-L12-v2`.

In [3]:
import re, nltk, torch
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util

nltk.download("punkt_tab", quiet=True)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

W_PAGE, W_FOCUS, W_POSITION = 0.40, 0.45, 0.15
NEEDS_CONTEXT = re.compile(r"^(And|But|So|Or|Also|However|Then|Here|There|This|These|That|It|They)\b")

def candidate_sentences(text, lo=70, hi=155):
    out = []
    for para in str(text).split("\n"):          # line breaks often separate headings from text
        for s in sent_tokenize(para.strip()):
            s = s.strip()
            if lo <= len(s) <= hi and s[0].isupper() and s[-1] in ".!?" and not NEEDS_CONTEXT.match(s):
                out.append(s)
    return out

def embedding_meta(text, title="", query=""):
    cands = candidate_sentences(text)
    if not cands:
        return "", 0.0
    E = embedder.encode(cands, convert_to_tensor=True, normalize_embeddings=True)
    page = torch.nn.functional.normalize(E.mean(0), dim=0)
    focus = embedder.encode(query or title or cands[0], convert_to_tensor=True, normalize_embeddings=True)
    position = torch.tensor([1 - i / len(cands) for i in range(len(cands))], device=E.device)
    score = W_PAGE * util.cos_sim(page, E)[0] + W_FOCUS * util.cos_sim(focus, E)[0] + W_POSITION * position
    best = int(score.argmax())
    return cands[best], round(float(score[best]), 3)

results = [embedding_meta(c, t, q) for c, t, q in zip(df["Content"], df["Title"], df["Top query"])]
df["Embedding meta"] = [r[0] for r in results]
df["Embedding score"] = [r[1] for r in results]
df[["URL", "Top query", "Embedding meta", "Embedding score"]]

,URL,Top query,Embedding meta,Embedding score
0,https://www.mlforseo.com/machine-learning-tuto...,what is fuzzy matching,"Fuzzy matching is a form of string matching, a...",0.805
1,https://www.mlforseo.com/machine-learning-tuto...,bertopic seo,The class-based TF-IDF step is worth understan...,0.590
2,https://www.mlforseo.com/machine-learning-impl...,keybert keyword clustering,"In this guide, I’ll show you how to work with ...",0.829
3,https://www.mlforseo.com/machine-learning-impl...,automate meta descriptions,"In this tutorial, we’ll automatically optimize...",0.621
4,https://www.mlforseo.com/glossary-term/bert-bi...,what is bert,"BERT is a language model developed by Google, ...",0.794


## 3. A small open LLM writes the description (abstractive)

A 1.7-billion-parameter model is tiny by LLM standards, but it's plenty for a two-sentence summary. The prompt does three jobs:
it **grounds** the model (only use facts in the page text), it sets the **format** (length, no quotes or emojis), and it passes the **title and top query** so the description matches what people search for.

We use greedy decoding (`do_sample=False`) so the same page always gets the same description, and we only send the first ~600 words,
which keeps it fast and is where most pages say what they're about. Small models are poor at counting characters, so if the first answer is over 155 characters we ask once more and tell it how long it was. Qwen3 has a "thinking" mode, which we switch off with `enable_thinking=False`.

Want something else? Any chat model on Hugging Face with a chat template works here, for example `Qwen/Qwen2.5-1.5B-Instruct`, `HuggingFaceTB/SmolLM2-1.7B-Instruct`
or a gated Llama/Gemma model once you've accepted its licence.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_ID = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype="auto", device_map="auto")

SYSTEM_PROMPT = (
    "You write meta descriptions for web pages. Use only facts stated in the page text. "
    "Write one or two sentences, 120-155 characters in total, plain text, no quotes, no hashtags, no emojis. "
    "If a target query is given and it fits the page naturally, include it."
)

def llm_meta(text, title="", query="", max_words=600):
    page = " ".join(str(text).split()[:max_words])
    user = f"Page title: {title}\nTarget query: {query or 'none'}\n\nPage text:\n{page}\n\nWrite the meta description."
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=80, do_sample=False)
    desc = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip().strip('"')
    # Small models are bad at counting characters. If it's too long, ask once more, with the number.
    if len(desc) > 155:
        messages += [{"role": "assistant", "content": desc},
                     {"role": "user", "content": f"That is {len(desc)} characters. Rewrite it in under 150 characters, same facts."}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
        out = llm.generate(**inputs, max_new_tokens=80, do_sample=False)
        desc = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip().strip('"')
    return desc

df["LLM meta"] = [llm_meta(c, t, q) if len(str(c).split()) >= 40 else ""
                  for c, t, q in zip(df["Content"], df["Title"], df["Top query"])]
df[["URL", "LLM meta"]]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:04<00:04,  4.11s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.02s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.33s/it]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,URL,LLM meta
0,https://www.mlforseo.com/machine-learning-tuto...,Fuzzy matching is a string matching technique ...
1,https://www.mlforseo.com/machine-learning-tuto...,"This post explains BERTopic, a modern topic mo..."
2,https://www.mlforseo.com/machine-learning-impl...,This guide explains how to use KeyBERT for key...
3,https://www.mlforseo.com/machine-learning-impl...,Automate your SEO metadata with FuzzyWuzzy and...
4,https://www.mlforseo.com/glossary-term/bert-bi...,BERT is a transformer-based language model dev...


## 4. Check both versions side by side

In [5]:
import re

MIN_CHARS, MAX_CHARS = 70, 155   # Google truncates by pixel width (~920px desktop); ~155 characters is a safe proxy

def check_description(desc, source_text, top_query=""):
    """Flags problems a human should look at. It does not prove a description is good."""
    flags = []
    n = len(desc)
    if n == 0:
        flags.append("EMPTY")
    elif n < MIN_CHARS:
        flags.append(f"SHORT ({n})")
    elif n > MAX_CHARS:
        flags.append(f"LONG ({n})")
    # numbers that appear in the description but nowhere in the page = classic hallucination
    src_numbers = set(re.findall(r"\d+(?:[.,]\d+)?", source_text))
    new_numbers = [x for x in re.findall(r"\d+(?:[.,]\d+)?", desc) if x not in src_numbers]
    if new_numbers:
        flags.append("NUMBER NOT IN PAGE: " + ", ".join(new_numbers))
    # capitalised words (names, brands, tools) that never appear in the page
    src_lower = source_text.lower()
    new_names = [w for w in re.findall(r"\b[A-Z][A-Za-z0-9]{2,}\b", desc)[1:] if w.lower() not in src_lower]
    if new_names:
        flags.append("NAME NOT IN PAGE: " + ", ".join(sorted(set(new_names))))
    if top_query:
        q_words = [w for w in re.findall(r"\w+", top_query.lower()) if len(w) > 3]
        if q_words and not any(w in desc.lower() for w in q_words):
            flags.append("TOP QUERY MISSING")
    return "; ".join(flags) or "OK"

In [6]:
# Duplicate descriptions are what we're trying to get rid of in the first place, so check for them
def flag_duplicates(frame, col):
    dupes = frame[col].str.strip().str.lower().duplicated(keep=False) & frame[col].str.strip().ne("")
    frame.loc[dupes, col + " check"] = frame.loc[dupes, col + " check"].replace("OK", "") + "; DUPLICATE"
    frame[col + " check"] = frame[col + " check"].str.strip("; ")
    return frame

In [7]:
for col in ["Embedding meta", "LLM meta"]:
    df[col + " check"] = [check_description(d, s, q) for d, s, q in zip(df[col], df["Content"], df["Top query"])]
    df = flag_duplicates(df, col)

# How close is each description to the page it describes? (a low score = drifted off-topic)
page_vecs = embedder.encode(df["Content"].str[:5000].tolist(), convert_to_tensor=True, normalize_embeddings=True)
for col in ["Embedding meta", "LLM meta"]:
    vecs = embedder.encode(df[col].tolist(), convert_to_tensor=True, normalize_embeddings=True)
    df[col + " ~page"] = util.cos_sim(vecs, page_vecs).diagonal().cpu().numpy().round(2)

df[["URL", "Embedding meta", "Embedding meta check", "Embedding meta ~page",
    "LLM meta", "LLM meta check", "LLM meta ~page"]]

,URL,Embedding meta,Embedding meta check,Embedding meta ~page,LLM meta,LLM meta check,LLM meta ~page
0,https://www.mlforseo.com/machine-learning-tuto...,"Fuzzy matching is a form of string matching, a...",OK,0.74,Fuzzy matching is a string matching technique ...,LONG (164),0.86
1,https://www.mlforseo.com/machine-learning-tuto...,The class-based TF-IDF step is worth understan...,OK,0.56,"This post explains BERTopic, a modern topic mo...",LONG (157),0.79
2,https://www.mlforseo.com/machine-learning-impl...,"In this guide, I’ll show you how to work with ...",OK,0.79,This guide explains how to use KeyBERT for key...,LONG (162),0.81
3,https://www.mlforseo.com/machine-learning-impl...,"In this tutorial, we’ll automatically optimize...",OK,0.75,Automate your SEO metadata with FuzzyWuzzy and...,OK,0.79
4,https://www.mlforseo.com/glossary-term/bert-bi...,"BERT is a language model developed by Google, ...",OK,0.62,BERT is a transformer-based language model dev...,OK,0.80


## 5. Save and review

Open the CSV, read every flagged row, and pick (or blend) the version you like. Neither method knows your brand voice or your offer. That's still your job.

In [8]:
df.drop(columns=["Content"]).to_csv("output_modern.csv", index=False)
print("Saved output_modern.csv with", len(df), "rows")

Saved output_modern.csv with 5 rows
